In [44]:
explanations = ["shap", "sage"]
estimators = ["kernel", "permutation"]
methods = ["cte", "cte_with_predictions", "cte_stratified"]

all_combinations = []
for explanation in explanations:
    for estimator in estimators:
        for method in methods:
            all_combinations.append(f'{method}_{explanation}_{estimator}')

all_combinations

['cte_shap_kernel',
 'cte_with_predictions_shap_kernel',
 'cte_stratified_shap_kernel',
 'cte_shap_permutation',
 'cte_with_predictions_shap_permutation',
 'cte_stratified_shap_permutation',
 'cte_sage_kernel',
 'cte_with_predictions_sage_kernel',
 'cte_stratified_sage_kernel',
 'cte_sage_permutation',
 'cte_with_predictions_sage_permutation',
 'cte_stratified_sage_permutation']

In [45]:
import time
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

from goodpoints import compress
from openxai.model import LoadModel
from openxai.dataloader import ReturnLoaders
import sage
import shap


np.random.seed(0)

data_name = 'compas'
shap_values_cte_path = f'../metadata/{data_name}/cte_shap_sage_results_0_10.npy'
times_info_path = f'../metadata/{data_name}/cte_shap_sage_times_0_10.csv'

shap_values_cte = np.load(shap_values_cte_path, allow_pickle=True).item()
times_info = pd.read_csv(times_info_path)

load data and model

In [62]:
_, loader_test = ReturnLoaders(data_name=data_name, download=False, batch_size=128)
X_test, y_test = loader_test.dataset.data, loader_test.dataset.targets.to_numpy()

model = LoadModel(data_name="compas", ml_model="ann", pretrained=True)
model.eval()

ArtificialNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=100, bias=True)
    (3): ReLU()
    (4): Linear(in_features=100, out_features=2, bias=True)
  )
)

###  calculate "ground truth"

In [ ]:
import time
import shap
import sage

class GroundTruthCalculator:
    def __init__(self, model, X_test, y_test, verbose=True):
        self.model = model
        self.X_test = X_test
        self.y_test = y_test
        self.verbose = verbose
        self.results = {}

    def compute_shap(self, estimator, seed=0):
        if self.verbose:
            print(f"Calculating SHAP ({estimator})...")
        start = time.time()
        if estimator == "kernel":
            explainer = shap.KernelExplainer(lambda x: self.model.predict_proba(x)[:, 1], self.X_test, seed=seed)
        elif estimator == "permutation":
            masker = shap.maskers.Independent(self.X_test, max_samples=self.X_test.shape[0])
            explainer = shap.PermutationExplainer(lambda x: self.model.predict_proba(x)[:, 1], masker, seed=seed)
        shap_values = explainer(self.X_test)
        elapsed = time.time() - start
        self.results[f"shap_{estimator}"] = {
            "values": shap_values.values,
            "time": elapsed
        }

    def compute_sage(self, estimator, seed=0):
        if self.verbose:
            print(f"Calculating SAGE ({estimator})...")
        start = time.time()
        imputer = sage.MarginalImputer(self.model.predict_proba, self.X_test)
        explainer = (sage.KernelEstimator if estimator == "kernel" else sage.PermutationEstimator)(imputer, loss="cross entropy", random_state=seed)
        sage_values = explainer(self.X_test, self.y_test, bar=False, verbose=False).values
        elapsed = time.time() - start
        self.results[f"sage_{estimator}"] = {
            "values": sage_values,
            "time": elapsed
        }

    def run_all(self, seed=0):
        self.compute_sage("kernel", seed)
        self.compute_sage("permutation", seed)
        self.compute_shap("permutation", seed)
        self.compute_shap("kernel", seed)
        if self.verbose:
            print("Ground truth calculation complete.")
        return self.results


In [64]:
gt_calc = GroundTruthCalculator(model, X_test, y_test)
gt_calc.y_test

array([0, 1, 0, ..., 1, 1, 0])

In [65]:
model.predict_proba(X_test)

array([[9.9825794e-01, 1.7420230e-03],
       [7.6656113e-09, 1.0000000e+00],
       [7.2401837e-03, 9.9275982e-01],
       ...,
       [1.4090156e-05, 9.9998593e-01],
       [1.5039637e-03, 9.9849606e-01],
       [6.6376263e-01, 3.3623737e-01]], dtype=float32)

In [61]:
gt_calc = GroundTruthCalculator(model, X_test, y_test)
ground_truth_results = gt_calc.run_all()

Calculating SAGE (kernel)...


ValueError: predictions are not valid probabilities

In [33]:
def metric_mae(x, y):
    return np.mean(np.abs(x-y))

In [37]:
mean_mae = {}
for key in all_combinations:
    shap_values = shap_values_cte[key]
    mean_mae = 0
    print(shap_values)
        


[array([[-8.28215414e-02, -1.82548900e-02, -7.59566310e-01, ...,
        -7.96919535e-03,  6.26854423e-04, -1.41294490e-02],
       [ 3.69153427e-02,  1.69422308e-02,  4.21545324e-02, ...,
        -2.80780594e-04, -2.78747392e-04,  7.90426818e-03],
       [ 4.19843537e-02,  4.22017870e-02, -1.82285285e-02, ...,
        -6.06631445e-03, -7.74200990e-04,  2.51917364e-02],
       ...,
       [-5.63551315e-02,  2.36076747e-02,  9.92399138e-02, ...,
         2.25712786e-02,  1.03346291e-02, -1.04747553e-03],
       [-3.11784395e-02,  2.46363690e-02,  1.06095735e-01, ...,
         1.23639504e-02, -1.14633165e-03, -4.71572256e-04],
       [ 4.24053173e-02,  7.31251501e-02, -5.57144255e-01, ...,
         7.21603304e-02, -2.43259245e-02, -1.04889871e-01]]), array([[-8.35867522e-02, -1.39439841e-02, -7.51726656e-01, ...,
        -5.52302472e-03, -4.17045070e-05, -1.22644702e-02],
       [ 4.42329033e-02,  1.56237276e-02,  3.41521871e-02, ...,
        -1.48494380e-03, -6.77611989e-05,  7.62590559